In [6]:
import os
import re
import math
import shutil
import subprocess

from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.dates import DateFormatter
from dateutil.tz import tzutc, tzlocal
from scipy import stats
from scipy.optimize import brentq, curve_fit, fsolve


import sys
import signal
import tempfile
import time
from collections import Counter
from IPython.display import clear_output

In [7]:
%load_ext autoreload
%autoreload 2

import official_GSSHA_Python_functions as gf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Load the paths for GSSHA model folder and executables

In [8]:
cwd = os.getcwd()
script_dir = Path.cwd() 

GSSHA_prj_name = "Waialua_FIM_testing"
model_dir = script_dir / GSSHA_prj_name
GSSHA_executables = script_dir / 'GSSHA_applications'

test_input_flow_results_dir = model_dir / "TEST_INPUT_FLOW_SIMULATIONS"

xys_tsf_iteration_file_name = GSSHA_prj_name + "_ITERATION"
xys_tsf_iteration_value = "155.0"

# Force shutdown of any GSSHA model running in model folder

In [9]:
 # Use  if you're in Jupyter or interactive session
process = gf.GSSHA_auto_shutdown(model_dir)

# When you need to force shutdown:
gf.force_shutdown_gssha(
    process=process,
    model_dir=model_dir
)

GSSHA started with process ID 13008
Project file: Waialua_FIM_testing.prj
Force-stopping GSSHA process tree for PID 13008...
SUCCESS: The process with PID 13008 (child process of PID 29520) has been terminated.
GSSHA stopped and gssha.exe is unlocked.


True

# Run flow testing for model development

In [16]:
FLOW_TEST = [2]
tot_time = 1000

In [17]:
gf.cleanup_model_dir(model_dir)

for flow in FLOW_TEST:
    new_value = float(flow)
    cfs_flow_for_filenames = round(flow*35.31467)
    
    output_file = (
        test_input_flow_results_dir
        / f"{GSSHA_prj_name}_OUTPUT-input-flow-cfs-{cfs_flow_for_filenames}.tsf"
    )
    
    if output_file.exists():
        print(f"{output_file.name} already exists. Skipping...")
        continue

    
    df_prj = gf.read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
    

    #ITERATION FILES
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = xys_tsf_iteration_file_name + ".tsf", save_filename = GSSHA_prj_name + "_bc.tsf", old_value =xys_tsf_iteration_value, new_value = new_value)
    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = xys_tsf_iteration_file_name + ".xys", save_filename = GSSHA_prj_name + ".xys", old_value = xys_tsf_iteration_value, new_value = new_value)
    ##UPDATE THE ITERATION W THE NEW XYS AND TSF FILE

    #!!!!write a function to copy with specified cfs flows in name!!!

    #SAVE TEXT XYS file for knowing the input flows for model run
    #These files are for the 

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + "_bc.tsf",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-input-flow-cfs-" + str(cfs_flow_for_filenames) + ".tsf"
    )


    

    df_prj.loc['TOT_TIME', 'Value'] = tot_time
    
    df_prj = df_prj.reset_index()


    #UPDATE PRJ FILE WHICH CONTAINS THE PATHS
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )


    gf.copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    
    #this function allows you to see when model convergence occurs so you can quit the run
    return_code = gf.run_gssha_convergence_view(
    MODEL_DIR=model_dir,
    PROJECT_FILE=GSSHA_prj_name + ".prj",
    DEP_FILE=GSSHA_prj_name + ".dep",
    cell_size=10,
    dep_check_seconds=15,
    display_last_n=10
)
    # move_and_rename_gssha_output(MODEL_DIR, RESULTS_DIR, output_description = "TEST_RUN" + first_date, extension = "otl")
    gf.cleanup_model_dir(model_dir)

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".dep",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-timeseries-depth-m-" + str(cfs_flow_for_filenames) + ".dep"
    )

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".gfl",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-maxflood-dep-m-" + str(cfs_flow_for_filenames) + ".gfl"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".oqc",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT_USGS-STATS-location-cms-" + str(cfs_flow_for_filenames) + ".oqc"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".ows",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT_WSE-active-USGS-gauge-m-" + str(cfs_flow_for_filenames) + ".ows"
    )


GSSHA simulation completed normally. The q window closed automatically.

Final DEP change results:


,timestep,cumulative_change,max_positive_change_per_cell,inundated_area_change
23,720.5,-0.006255,0.007939,0.0
24,750.5,0.084802,0.023756,0.0
25,780.5,0.532861,0.020769,0.0
26,810.5,-0.147810,0.006657,100.0
27,840.5,0.321875,0.011797,0.0
28,870.5,-0.058144,0.020603,-100.0
29,900.5,-0.266186,0.017691,0.0
30,930.5,0.150890,0.025388,0.0
31,960.5,-0.017554,0.007351,0.0
32,990.5,0.177525,0.023874,100.0


✔️ Cleaned up model folder.


# Read Results from calibration/senstivity and run model for final stage height maps (datum offset with model parameters: input flow, simulation time)

## ***REFER TO POST PROCESSS NOTEBOOK FOR Model inpuTS FROM THE TEST SIMULATION ABOVE**

In [10]:
offset = 13.835
test_flow_results = pd.read_csv(test_input_flow_results_dir /(str(offset) + "modeling_test_results.csv"))
test_flow_results = test_flow_results.set_index('gauge_readings_ft')

final_stage_maps_dir = model_dir / ("FINAL_STAGE_MAPS_OFFSET" + str(offset))
final_stage_maps_dir.mkdir(
    parents=True,
    exist_ok=True,
)
test_flow_results.to_csv(final_stage_maps_dir / "final_model_input_flows.csv")
print(final_stage_maps_dir)

C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waialua_FIM_testing\FINAL_STAGE_MAPS_OFFSET13.835


In [ ]:
gf.cleanup_model_dir(model_dir)

for gauge_reading in test_flow_results.index:
    input_flow = test_flow_results['input_flows'][gauge_reading].round(0)
    total_run_time = test_flow_results['convergence_time'][gauge_reading]

    
    new_value = float(input_flow)
    cfs_flow_for_filenames = round(input_flow*35.31467)
    
    output_file = (
        final_stage_maps_dir
        / f"{GSSHA_prj_name}_{gauge_reading}_OUTPUT-input-flow-cfs-{cfs_flow_for_filenames}.tsf"
    )
    
    if output_file.exists():
        print(f"{output_file.name} already exists. Skipping...")
        continue

    
    df_prj = gf.read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
    

    #ITERATION FILES
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = xys_tsf_iteration_file_name + ".tsf", save_filename = GSSHA_prj_name + "_bc.tsf", old_value =xys_tsf_iteration_value, new_value = new_value)
    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = xys_tsf_iteration_file_name + ".xys", save_filename = GSSHA_prj_name + ".xys", old_value = xys_tsf_iteration_value, new_value = new_value)
    ##UPDATE THE ITERATION W THE NEW XYS AND TSF FILE

    #!!!!write a function to copy with specified cfs flows in name!!!

    #SAVE TEXT XYS file for knowing the input flows for model run
    #These files are for the 

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + "_bc.tsf",
        output_folder=final_stage_maps_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT-input-flow-cfs-" + str(cfs_flow_for_filenames) + ".tsf"
    )


    

    df_prj.loc['TOT_TIME', 'Value'] = total_run_time
    
    df_prj = df_prj.reset_index()


    #UPDATE PRJ FILE WHICH CONTAINS THE PATHS
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )


    gf.copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    gf.run_gssha(model_dir, PROJECT_FILE = GSSHA_prj_name + ".prj")
    gf.cleanup_model_dir(model_dir)

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".dep",
        output_folder=final_stage_maps_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT-timeseries-depth-m-" + str(cfs_flow_for_filenames) + ".dep"
    )

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".gfl",
        output_folder=final_stage_maps_dir,
        new_filename=GSSHA_prj_name+"_" + str(gauge_reading) + "_OUTPUT-maxflood-dep-m-" + str(cfs_flow_for_filenames) + ".gfl"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".oqc",
        output_folder=final_stage_maps_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT_USGS-STATS-location-cms-" + str(cfs_flow_for_filenames) + ".oqc"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".ows",
        output_folder=final_stage_maps_dir,
        new_filename=GSSHA_prj_name +"_" + str(gauge_reading) + "_OUTPUT_WSE-active-USGS-gauge-m-" + str(cfs_flow_for_filenames) + ".ows"
    )


✔️ Cleaned up model folder.
✅ Successfully wrote: C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waialua_FIM_testing\Waialua_FIM_testing.prj
✔️ GSSHA applications copied to model folder.
📦 Running GSSHA:
  Executable: C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waialua_FIM_testing\gssha.exe
  Project File: C:\Users\bgorberg\Documents\GitHub\Flood_Stage_Maps\RUN_GSSHA\Waialua_FIM_testing\Waialua_FIM_testing.prj
